# 01 · Your first LLM call, with LangChain

**AI Fundamentals in 3 Hours** · Data Sense

By the end of this notebook you will have:

1. Made your first call to a language model
2. Understood the message objects every LLM conversation is made of
3. Seen what a system prompt actually changes
4. Proved to yourself that **the model has no memory:** and then fixed it
5. Swapped the model provider by changing **one string**

### Why LangChain, from the very first line

Every provider. OpenAI, Anthropic, Google, Mistral, Groq, a local model, has its own SDK,
its own response shape, its own name for everything. Learn one and you have learned one.

LangChain gives you **one interface** over all of them, and the same interface carries all
the way up to RAG, tools and agents. Learn it once here, and the rest of the workshop, and most of the AI work you do afterwards, is the same handful of objects.

```
pip install -r ../requirements.txt
```


In [ ]:
# --- run this first, in every notebook ---
import os, json
from pathlib import Path

# read keys out of .env (works from the repo root or from notebooks/)
for candidate in [Path(".env"), Path("../.env")]:
    if candidate.exists():
        for line in candidate.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break

from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1-mini"          # provider:model - change this one string to switch providers
model = init_chat_model(MODEL, temperature=0)

import textwrap
def wrap(text, width=88):
    """Print long text wrapped, so answers stay readable on a projector."""
    print(textwrap.fill(str(text), width=width))

assert os.environ.get("OPENAI_API_KEY"), "No API key found - check your .env file"
print("ready |", MODEL)

## 1. The smallest possible call

One line. `invoke` is the verb for *everything* in LangChain, models, tools, retrievers,
agents all have it. Learn this verb and you have learned most of the library.

In [ ]:
response = model.invoke("Explain what an API is, in two sentences, for someone who has never coded.")


wrap(response.content)

## 2. What actually came back

Not a string, an **`AIMessage`** object. The text is in `.content`, and useful metadata
sits alongside it.

`usage_metadata` is the one people ignore and shouldn't: it is your bill, itemised. Note
that it has the **same shape no matter which provider you use**. That is the first real
payoff of the framework.

In [ ]:
print("type            :", type(response).__name__)
print("content         :", response.content[:70], "...")
print()
u = response.usage_metadata
print("input tokens    :", u["input_tokens"])
print("output tokens   :", u["output_tokens"])
print("total tokens    :", u["total_tokens"])
print()
print("finish_reason   :", response.response_metadata.get("finish_reason"))
print("model_name      :", response.response_metadata.get("model_name"))

### `finish_reason` is worth memorising

| value | meaning |
|---|---|
| `stop` | finished naturally. This is what you want. |
| `length` | your token limit cut it off mid-sentence. Very common beginner bug. |
| `tool_calls` | it wants you to run a function. Notebook 05. |
| `content_filter` | the safety system blocked it. |

Let's cause a `length` finish deliberately, so you recognise it when it happens for real.

In [ ]:
short_model = init_chat_model(MODEL, max_tokens=30)
truncated = short_model.invoke("Write a 500 word essay about the Indian monsoon.")

print(truncated)

wrap(truncated.content)
print()
print("finish_reason:", truncated.response_metadata.get("finish_reason"), " <-- cut off, not finished")

## 3. Messages: the three roles

A conversation is an **ordered list of messages**. There are three you will use constantly:

| LangChain class | role | written by |
|---|---|---|
| `SystemMessage` | `system` | **you, the developer:** standing instructions |
| `HumanMessage` | `user` | **your end user:** this turn's request |
| `AIMessage` | `assistant` | **the model:** its reply |

There is nothing else. Everything the model knows on a given turn is in that list.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

reply = model.invoke([
    SystemMessage("You are a support agent for Nimbus Retail. Answer in at most 2 short sentences."),
    HumanMessage("My order hasn't arrived yet. What do I do?"),
])

wrap(reply.content)
print()
print("the reply is a:", type(reply).__name__)

**System Prompt:** Behave like a Robot

**Human Question:** What is API?

`[M1, M2, M3, M4 ....... M10]`


### LangChain also accepts plain dicts and tuples

These three are identical. Use whichever you find readable, the class form is the clearest
for beginners, the dict form is what you will see in most documentation.

In [ ]:
a = [SystemMessage("Be terse."), HumanMessage("Capital of Karnataka?")]
b = [{"role": "system", "content": "Be terse."}, {"role": "user", "content": "Capital of Karnataka?"}]
c = [("system", "Be terse."), ("user", "Capital of Karnataka?")]

for style, msgs in [("classes", a), ("dicts", b), ("tuples", c)]:
    print(f"{style:<9}->", model.invoke(msgs).content)

## 4. The system prompt is your biggest lever

Same model, same question, three different products.

In [ ]:
question = "My order hasn't arrived yet. What do I do?"

personas = {
    "terse support agent":
        "You are a support agent. Answer in at most 2 short sentences. Never apologise more than once.",
    "over-friendly bot":
        "You are an extremely enthusiastic assistant. Use lots of exclamation marks and emoji.",
    "legal-cautious":
        "You are a compliance officer. Never promise a refund or a timeline. Be precise and hedged.",
}

for name, system_prompt in personas.items():
    out = model.invoke([SystemMessage(system_prompt), HumanMessage(question)])
    print(f"--- {name} ---")
    wrap(out.content)
    print()

> ⚠️ **Security note.** Never build your system prompt by pasting in text your user typed.
> If a user can write into your system prompt, they control your application. That is
> **prompt injection**, and it is the number one AI security bug.
>
> The system prompt is the **job description**. The user message is **today's ticket**.
> Never let the ticket rewrite the job description.

## 5. The model has no memory

This is the part that surprises everyone.

In [ ]:
wrap("turn 1: " + model.invoke("My name is Priya and I work in Hyderabad.").content)
print()
wrap("turn 2: " + model.invoke("What is my name and where do I work?").content)

It has no idea. There is **no session**, no user id, no server-side memory. Every request
arrives as a complete stranger.

ChatGPT feels like it remembers you because *the app* resends the whole conversation every
single time. So let's do what the app does, pass the history ourselves.

In [ ]:
history = [
    HumanMessage("My name is Priya and I work in Hyderabad."),
    AIMessage("Nice to meet you, Priya!"),          # what it said last turn
    HumanMessage("What is my name and where do I work?"),
]

wrap(model.invoke(history).content)

**That is the entire trick behind every chatbot you have ever used.**

Two consequences you now have to design around:

1. **Conversation history is your job.** You store it, append to it, and resend it.
2. **Long conversations cost more per turn:** turn 20 resends turns 1 to 19, so you pay
   for them again. The bill grows with the conversation.

### Doing it by hand

Here is memory written out manually, so you can see there is no magic in it.

In [ ]:
class Chat:
    """Manual conversation memory. Twelve lines. This is all a chatbot is."""

    def __init__(self, system_prompt: str):
        self.messages = [SystemMessage(system_prompt)]
        self.tokens_in = 0
        self.tokens_out = 0

    def say(self, text: str) -> str:
        self.messages.append(HumanMessage(text))
        reply = model.invoke(self.messages)
        self.messages.append(reply)                 # <-- this line IS the memory
        self.tokens_in  += reply.usage_metadata["input_tokens"]
        self.tokens_out += reply.usage_metadata["output_tokens"]
        return reply.content


chat = Chat("You are a concise assistant for Nimbus Retail. Keep answers under 40 words.")
wrap(chat.say("Hi, I'm Priya."))
wrap(chat.say("I ordered a table lamp last Tuesday and it hasn't arrived."))
wrap(chat.say("What did I order, again?"))

In [ ]:
for m in chat.messages:
    print(f"{type(m).__name__:<14}| {m.content[:76]}")

print()
print(f"input tokens across the conversation : {chat.tokens_in:,}")
print(f"output tokens                        : {chat.tokens_out:,}")
print()
print("The input count grows every turn - you resend everything, every time.")

## 6. Now let LangChain do it for you

You have seen the mechanism. Here is the same thing with a **checkpointer**: LangChain
stores the history, keyed by a `thread_id`, and resends it for you.

Every user gets their own `thread_id`. That is how one application serves many conversations.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(
    model=model,
    tools=[],                                        # no tools yet - notebook 05
    system_prompt="You are a concise assistant for Nimbus Retail. Keep answers under 40 words.",
    checkpointer=InMemorySaver(),                    # <-- memory, handled for you
)

priya = {"configurable": {"thread_id": "priya"}}

def ask(text, cfg):
    out = assistant.invoke({"messages": [HumanMessage(text)]}, config=cfg)
    return out["messages"][-1].content

wrap(ask("Hi, I'm Priya. I ordered a table lamp.", priya))
wrap(ask("What did I order?", priya))

### Separate threads are genuinely separate

Swap the `thread_id` and the memory is gone, exactly as it should be for a different user.

In [ ]:
rahul = {"configurable": {"thread_id": "rahul"}}

wrap("rahul asks : " + ask("What did I order?", rahul))
print()
wrap("priya asks : " + ask("And what was my name again?", priya))

> **In production**, swap `InMemorySaver()` for a database-backed checkpointer and the
> conversation survives a restart. Your code does not change, only that one line.

## 7. The payoff: switching providers

This is the argument for learning LangChain rather than one vendor's SDK.

Everything above, messages, `invoke`, `usage_metadata`, the checkpointer, and every agent
you build later, works identically if you change the model string.

In [ ]:
print("Same code, different model. Only the string changes:\n")

for name in ["openai:gpt-4.1-mini", "openai:gpt-4.1-nano"]:
    m = init_chat_model(name, temperature=0)
    out = m.invoke([SystemMessage("Answer in exactly five words."), HumanMessage("What is an LLM?")])
    print(f"  {name:<24}-> {out.content}")
    print(f"  {'':<24}   {out.usage_metadata['input_tokens']} in / {out.usage_metadata['output_tokens']} out")

print()
print("These would work the same way with an API key for each provider:")
for name in ["anthropic:claude-sonnet-4-5", "google_genai:gemini-2.0-flash",
             "groq:llama-3.3-70b-versatile", "ollama:llama3.2"]:
    print("   ", name)

No rewrite. No new SDK. No new response shape to learn. That is the whole pitch.

## 8. What LangChain is actually doing

A framework should be transparent, not magic. Every provider ultimately receives JSON over
HTTPS, and you can see exactly what LangChain built for you.

In [ ]:
payload = model._get_request_payload([
    SystemMessage("You are a terse tutor."),
    HumanMessage("What is an API?"),
])

print(json.dumps(payload, indent=2)[:600])
print("\n... this is the HTTP request body LangChain sends on your behalf.")

There it is `model`, `messages`, `role`, `content`. Exactly the five parts from the
slides. LangChain is not hiding the API from you; it is giving you one shape for all of them.

*(`_get_request_payload` has a leading underscore because it is internal. It is here to
prove a point, not for production use.)*

## 9. Your turn

1. **Change the persona.** Rewrite the system prompt so the assistant answers only in bullet
   points and refuses anything not about Nimbus Retail orders. Then try to talk it out of it.

2. **Break it deliberately.** Set `max_tokens=20` and watch `finish_reason` become `length`.

3. **Two users, one agent.** Add a third `thread_id` and convince yourself the histories
   never leak between them.

4. **Count the cost.** `gpt-4.1-mini` costs \$0.40 per million input tokens and \$1.60 per
   million output. Using `chat.tokens_in` and `chat.tokens_out`, what did that conversation
   cost? What would 10,000 a day cost? Notebook 02 does this properly.


In [ ]:
# your turn - scratch cell
